# BackPaS: técnicas pendientes en paralelo

Cada sesión debe usar un trabajo distinto de `configs/pending_augmentation_jobs.json`. Ya están cubiertos baseline/full, polarity y permutation para MVC, MIS y CA; esta matriz incluye solo las técnicas pendientes. La sesión genera su dataset, lo valida, prepara BackPaS y entrena con checkpoints en Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

REPO = '/content/backbone-pbo'
DRIVE_ROOT = '/content/drive/MyDrive/PBO-Tesis'
JOB = 'MVC/selective-polarity/seed-0'  # Distinto en cada sesión


In [ ]:
# Obtener el código actualizado y los datasets base desde Drive.
import os, subprocess, shutil, tarfile, glob, json
if not os.path.exists(REPO):
    subprocess.run(['git','clone','https://github.com/jgomezaTI/PBO-DA.git', REPO], check=True)
else:
    subprocess.run(['git','-C', REPO, 'pull', '--ff-only'], check=True)

DATA_ROOT = f'{DRIVE_ROOT}/datasets'
for name in ['MVC', 'CA']:
    archive = f'{DATA_ROOT}/{name}/{name}.tar.gz'
    if os.path.exists(archive) and not os.path.exists(f'{REPO}/data/backpas/{name}'):
        with tarfile.open(archive, 'r:gz') as tar:
            tar.extractall(f'{REPO}/data/backpas')
mis_dir = f'{DATA_ROOT}/MIS'
mis_archive = f'{mis_dir}/MIS.tar.gz'
parts = sorted(glob.glob(f'{mis_dir}/MIS.tar.gz.part-*'))
if not os.path.exists(mis_archive) and parts:
    with open(mis_archive, 'wb') as output:
        for part in parts:
            with open(part, 'rb') as source:
                shutil.copyfileobj(source, output)
if os.path.exists(mis_archive) and not os.path.exists(f'{REPO}/data/backpas/MIS'):
    with tarfile.open(mis_archive, 'r:gz') as tar:
        tar.extractall(f'{REPO}/data/backpas')

subprocess.run(['python','-m','pip','install','-e','.[ml]'], cwd=REPO, check=True)


In [ ]:
# Ver la lista pendiente y ejecutar exactamente el trabajo seleccionado.
from pathlib import Path
matrix = json.loads(Path(REPO, 'configs/pending_augmentation_jobs.json').read_text())
print('Trabajos pendientes:', len(matrix['jobs']))
subprocess.run(['python','scripts/cloud_run.py','--repo',REPO,'--matrix','configs/pending_augmentation_jobs.json','--job',JOB,'--drive-root',DRIVE_ROOT], cwd=REPO, check=True)
